In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("librerie caricate correttamente")


librerie caricate correttamente


In [2]:
df_gen = pd.read_csv('../data/raw/nuclear-energy-generation.csv')
df_owid = pd.read_csv('../data/raw/owid-energy-data.csv')

print("=== DATASET 1 — nuclear-energy-generation ===")
print(f"Righe: {len(df_gen)} | Colonne: {len(df_gen.columns)}")
print(f"Colonne: {list(df_gen.columns)}")

print("\n=== DATASET 2 — owid-energy-data ===")
print(f"Righe: {len(df_owid)} | Colonne: {len(df_owid.columns)}")
print(f"Colonne: {list(df_owid.columns)}")

=== DATASET 1 — nuclear-energy-generation ===
Righe: 10064 | Colonne: 4
Colonne: ['Entity', 'Code', 'Year', 'Nuclear']

=== DATASET 2 — owid-energy-data ===
Righe: 23232 | Colonne: 130
Colonne: ['country', 'year', 'iso_code', 'population', 'gdp', 'biofuel_cons_change_pct', 'biofuel_cons_change_twh', 'biofuel_cons_per_capita', 'biofuel_consumption', 'biofuel_elec_per_capita', 'biofuel_electricity', 'biofuel_share_elec', 'biofuel_share_energy', 'carbon_intensity_elec', 'coal_cons_change_pct', 'coal_cons_change_twh', 'coal_cons_per_capita', 'coal_consumption', 'coal_elec_per_capita', 'coal_electricity', 'coal_prod_change_pct', 'coal_prod_change_twh', 'coal_prod_per_capita', 'coal_production', 'coal_share_elec', 'coal_share_energy', 'electricity_demand', 'electricity_demand_per_capita', 'electricity_generation', 'electricity_share_energy', 'energy_cons_change_pct', 'energy_cons_change_twh', 'energy_per_capita', 'energy_per_gdp', 'fossil_cons_change_pct', 'fossil_cons_change_twh', 'fossil_e

In [3]:
print("=== Prime 5 righe ===")
print(df_gen.head())

print("\n=== Ultime 5 righe ===")
print(df_gen.tail())

print("\n=== Range temporale ===")
print(f"Anno minimo: {df_gen['Year'].min()}")
print(f"Anno massimo: {df_gen['Year'].max()}")

print("\n=== Quanti paesi unici? ===")
print(f"Paesi/entità: {df_gen['Entity'].nunique()}")

print("\n=== Valori mancanti per colonna ===")
print(df_gen.isnull().sum())


=== Prime 5 righe ===
          Entity Code  Year  Nuclear
0  ASEAN (Ember)  NaN  2000      0.0
1  ASEAN (Ember)  NaN  2001      0.0
2  ASEAN (Ember)  NaN  2002      0.0
3  ASEAN (Ember)  NaN  2003      0.0
4  ASEAN (Ember)  NaN  2004      0.0

=== Ultime 5 righe ===
         Entity Code  Year  Nuclear
10059  Zimbabwe  ZWE  2020      0.0
10060  Zimbabwe  ZWE  2021      0.0
10061  Zimbabwe  ZWE  2022      0.0
10062  Zimbabwe  ZWE  2023      0.0
10063  Zimbabwe  ZWE  2024      0.0

=== Range temporale ===
Anno minimo: 1965
Anno massimo: 2025

=== Quanti paesi unici? ===
Paesi/entità: 249

=== Valori mancanti per colonna ===
Entity        0
Code       1091
Year          0
Nuclear       0
dtype: int64


In [4]:
# Selezioniamo solo le colonne che ci interessano
cols_nucleari = ['country', 'year', 'population', 'gdp',
                 'nuclear_electricity', 'nuclear_share_elec',
                 'nuclear_share_energy', 'nuclear_elec_per_capita']

df_nuclear = df_owid[cols_nucleari].copy()

print("=== Dataset nucleare estratto ===")
print(f"Righe: {len(df_nuclear)} | Colonne: {len(df_nuclear.columns)}")

print("\n=== Prime 5 righe ===")
print(df_nuclear.head())

print("\n=== Valori mancanti ===")
print(df_nuclear.isnull().sum())

print("\n=== Statistiche base ===")
print(df_nuclear.describe())

=== Dataset nucleare estratto ===
Righe: 23232 | Colonne: 8

=== Prime 5 righe ===
         country  year  population  gdp  nuclear_electricity  \
0  ASEAN (Ember)  2000         NaN  NaN                  0.0   
1  ASEAN (Ember)  2001         NaN  NaN                  0.0   
2  ASEAN (Ember)  2002         NaN  NaN                  0.0   
3  ASEAN (Ember)  2003         NaN  NaN                  0.0   
4  ASEAN (Ember)  2004         NaN  NaN                  0.0   

   nuclear_share_elec  nuclear_share_energy  nuclear_elec_per_capita  
0                 0.0                   NaN                      NaN  
1                 0.0                   NaN                      NaN  
2                 0.0                   NaN                      NaN  
3                 0.0                   NaN                      NaN  
4                 0.0                   NaN                      NaN  

=== Valori mancanti ===
country                        0
year                           0
population     

In [5]:
# Filtriamo solo paesi reali (hanno il codice ISO) ed escludiamo aggregati
df_paesi = df_gen[df_gen['Code'].notna()].copy()

# Top 10 paesi per produzione media storica
top10 = (df_paesi.groupby('Entity')['Nuclear']
         .mean()
         .sort_values(ascending=False)
         .head(10)
         .round(1))

print("=== Top 10 paesi per produzione nucleare media (TWh) ===")
print(top10)

print(f"\nTotale paesi con codice ISO: {df_paesi['Entity'].nunique()}")
print(f"Range anni dopo filtro: {df_paesi['Year'].min()} - {df_paesi['Year'].max()}")

=== Top 10 paesi per produzione nucleare media (TWh) ===
Entity
World                            1762.9
High-income countries            1568.8
Europe                            804.6
North America                     594.4
European Union (27)               561.3
United States                     533.3
Asia                              324.5
France                            270.3
Upper-middle-income countries     157.8
Russia                            154.8
Name: Nuclear, dtype: float64

Totale paesi con codice ISO: 224
Range anni dopo filtro: 1965 - 2025


In [6]:
import re

mask = df_gen['Code'].str.match(r'^[A-Z]{3}$', na=False)
df_paesi = df_gen[mask].copy()

# Vediamo tutti i codici unici che passano il filtro
codici = df_paesi[['Entity', 'Code']].drop_duplicates().sort_values('Code')

print(f"Totale entità con codice 3 lettere: {len(codici)}")
print("\n=== Prima lista completa ===")
print(codici.to_string())


Totale entità con codice 3 lettere: 210

=== Prima lista completa ===
                                 Entity Code
469                               Aruba  ABW
25                          Afghanistan  AFG
300                              Angola  AGO
192                             Albania  ALB
9347               United Arab Emirates  ARE
384                           Argentina  ARG
444                             Armenia  ARM
276                      American Samoa  ASM
360                 Antigua and Barbuda  ATG
636                           Australia  AUS
696                             Austria  AUT
757                          Azerbaijan  AZE
1481                            Burundi  BDI
1004                            Belgium  BEL
1089                              Benin  BEN
1457                       Burkina Faso  BFA
880                          Bangladesh  BGD
1396                           Bulgaria  BGR
820                             Bahrain  BHR
797                           

In [7]:
# Top 10 paesi reali per produzione media storica
top10 = (df_paesi.groupby('Entity')['Nuclear']
         .mean()
         .sort_values(ascending=False)
         .head(10)
         .round(1))

print("=== Top 10 paesi per produzione nucleare media storica (TWh) ===")
print(top10)

=== Top 10 paesi per produzione nucleare media storica (TWh) ===
Entity
United States     533.3
France            270.3
Russia            154.8
Japan             136.5
Germany            92.6
Ukraine            78.0
South Korea        76.8
China              73.5
Canada             65.0
United Kingdom     58.1
Name: Nuclear, dtype: float64


In [8]:
# Paesi con almeno un anno di produzione nucleare > 0
paesi_nucleari = (df_paesi[df_paesi['Nuclear'] > 0]
                  ['Entity']
                  .unique())

print(f"Paesi che hanno MAI prodotto energia nucleare: {len(paesi_nucleari)}")
print("\nLista:")
for p in sorted(paesi_nucleari):
    print(f"  {p}")

Paesi che hanno MAI prodotto energia nucleare: 36

Lista:
  Argentina
  Armenia
  Belarus
  Belgium
  Brazil
  Bulgaria
  Canada
  China
  Czechia
  Finland
  France
  Germany
  Hungary
  India
  Iran
  Italy
  Japan
  Kazakhstan
  Lithuania
  Mexico
  Netherlands
  Pakistan
  Romania
  Russia
  Slovakia
  Slovenia
  South Africa
  South Korea
  Spain
  Sweden
  Switzerland
  Taiwan
  Ukraine
  United Arab Emirates
  United Kingdom
  United States
